# 01 — Dataset Acquisition
**ClimateSense AI** · Debshata Choudhury (A2305223104) · Generative AI, Amity University

**Objective.** Download the four Kaggle datasets that the project is restricted to, record their
provenance, and inspect the actual files and schemas before any modelling.

| Name | Kaggle handle | Role |
|---|---|---|
| World Weather Repository | `nelgiriyewithana/global-weather-repository` | daily capital-city weather + air quality (anchor) |
| US Drought & Meteorological | `cdminix/us-drought-meteorological-data` | auxiliary drought benchmark / transfer learning |
| World Cities | `juanmah/world-cities` | coordinates, ISO codes, population |
| Berkeley Earth | `berkeleyearth/climate-change-earth-surface-temperature-data` | long-term temperature climatology |

Downloads are performed by `scripts/download_datasets.py` (kagglehub, cached, retried). The Kaggle
token is read from a local, git-ignored `.env`; it is never printed or committed.

In [1]:
import sys, json, warnings
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib
import matplotlib.pyplot as plt
from IPython.display import Image, display, Markdown
pd.set_option("display.max_columns", 60); pd.set_option("display.width", 220)
from climatesense.utils import read_json
import platform, torch, sklearn
print("Executed:", pd.Timestamp.now().strftime("%Y-%m-%d %H:%M"), "| Python", platform.python_version(),
      "| torch", torch.__version__, "| sklearn", sklearn.__version__, "| pandas", pd.__version__)
def show(p, width=850):
    display(Image(filename=str(ROOT / p), width=width))

Executed: 2026-10-25 18:05 | Python 3.13.7 | torch 2.14.1+cu126 | sklearn 1.9.1 | pandas 3.1.0


## 1. Provenance manifest
The manifest records handle, resolved version, files and byte sizes.

In [2]:
man = read_json(ROOT / "data/manifests/download_manifest.json")
print("downloaded at (UTC):", man["downloaded_at_utc"], "| kagglehub", man["kagglehub_version"])
rows = []
for k, v in man["datasets"].items():
    rows.append(dict(dataset=k, handle=v["handle"], version=v.get("version"), status=v["status"],
                     skipped_empty_versions=v.get("skipped_empty_versions"), n_files=len(v.get("files", [])),
                     total_MB=round(sum(f["bytes"] for f in v.get("files", [])) / 1e6, 1)))
pd.DataFrame(rows)

downloaded at (UTC): 2026-10-25T09:23:13.372500+00:00 | kagglehub 1.0.3


,dataset,handle,version,status,skipped_empty_versions,n_files,total_MB
0,weather,nelgiriyewithana/global-weather-repository,1105,ok,[1106],2,45.0
1,drought_us,cdminix/us-drought-meteorological-data,5,ok,None,4,2713.9
2,world_cities,juanmah/world-cities,12,ok,None,1,5.3
3,historical_temperature,berkeleyearth/climate-change-earth-surface-tem...,2,ok,None,5,600.6


The latest version of the weather repository (1106) was published as an empty archive on the
download date; the script detected this and fell back to version 1105, which is recorded above.

In [3]:
for k, v in man["datasets"].items():
    print(k); [print("   ", f["file"], f"{f['bytes']/1e6:.1f} MB") for f in v["files"]]

weather
    GlobalWeatherRepository.csv 45.0 MB
    state.db 0.0 MB
drought_us
    soil_data.csv 0.7 MB
    test_timeseries/test_timeseries.csv 258.4 MB
    train_timeseries/train_timeseries.csv 2196.4 MB
    validation_timeseries/validation_timeseries.csv 258.3 MB
world_cities
    worldcities.csv 5.3 MB
historical_temperature
    GlobalLandTemperaturesByCity.csv 532.8 MB
    GlobalLandTemperaturesByCountry.csv 22.7 MB
    GlobalLandTemperaturesByMajorCity.csv 14.1 MB
    GlobalLandTemperaturesByState.csv 30.8 MB
    GlobalTemperatures.csv 0.2 MB


## 2. Actual schemas (first rows) — no column names are assumed

In [4]:
w = pd.read_csv(ROOT / "data/raw/weather/GlobalWeatherRepository.csv", nrows=3)
print(w.shape[1], "columns"); w.T

41 columns


,0,1,2
country,Afghanistan,Albania,Algeria
location_name,Kabul,Tirana,Algiers
latitude,34.52,41.33,36.76
longitude,69.18,19.82,3.05
timezone,Asia/Kabul,Europe/Tirane,Africa/Algiers
last_updated_epoch,1715849100,1715849100,1715849100
last_updated,2024-05-16 13:15,2024-05-16 10:45,2024-05-16 09:45
temperature_celsius,26.6,19.0,23.0
temperature_fahrenheit,79.8,66.2,73.4
condition_text,Partly Cloudy,Partly cloudy,Sunny


In [5]:
display(pd.read_csv(ROOT / "data/raw/world_cities/worldcities.csv", nrows=3))
display(pd.read_csv(ROOT / "data/raw/historical_temperature/GlobalLandTemperaturesByCity.csv", nrows=3))
display(pd.read_csv(ROOT / "data/raw/drought_us/train_timeseries/train_timeseries.csv", nrows=3))

,city,city_ascii,lat,lng,country,iso2,iso3,admin_name,capital,population,id
0,Tokyo,Tokyo,35.6850,139.7514,Japan,JP,JPN,Tōkyō,primary,39105000,1392685764
1,Jakarta,Jakarta,-6.1753,106.8269,Indonesia,ID,IDN,Jakarta,primary,33756000,1360771077
2,Guangzhou,Guangzhou,23.1300,113.2600,China,CN,CHN,Guangdong,admin,26940000,1156237133


,dt,AverageTemperature,AverageTemperatureUncertainty,City,Country,Latitude,Longitude
0,1743-11-01,6.068,1.737,Århus,Denmark,57.05N,10.33E
1,1743-12-01,NaN,NaN,Århus,Denmark,57.05N,10.33E
2,1744-01-01,NaN,NaN,Århus,Denmark,57.05N,10.33E


,fips,date,PRECTOT,PS,QV2M,T2M,T2MDEW,T2MWET,T2M_MAX,T2M_MIN,T2M_RANGE,TS,WS10M,WS10M_MAX,WS10M_MIN,WS10M_RANGE,WS50M,WS50M_MAX,WS50M_MIN,WS50M_RANGE,score
0,1001,2000-01-01,0.22,100.51,9.65,14.74,13.51,13.51,20.96,11.46,9.50,14.65,2.20,2.94,1.49,1.46,4.85,6.04,3.23,2.81,NaN
1,1001,2000-01-02,0.20,100.55,10.42,16.69,14.71,14.71,22.80,12.61,10.18,16.60,2.52,3.43,1.83,1.60,5.33,6.13,3.72,2.41,NaN
2,1001,2000-01-03,3.65,100.15,11.76,18.49,16.52,16.52,22.73,15.32,7.41,18.41,4.03,5.33,2.66,2.67,7.53,9.52,5.87,3.66,NaN


## 3. Size, coverage and quality summary
Produced by `scripts/inspect_datasets.py` (full-file scans).

In [6]:
q = read_json(ROOT / "data/manifests/dataset_quality.json")
summ = {
 "weather": dict(rows=q["weather"]["rows"], cols=len(q["weather"]["columns"]), start=q["weather"]["date_min"], end=q["weather"]["date_max"],
                 dup_ids=q["weather"]["duplicate_ids"], countries_raw=q["weather"]["n_countries_raw"]),
 "world_cities": dict(rows=q["world_cities"]["rows"], cols=len(q["world_cities"]["columns"]), dup_ids=q["world_cities"]["duplicate_ids"]),
 "berkeley_by_city": dict(rows=q["berkeley_by_city"]["rows"], cols=len(q["berkeley_by_city"]["columns"]), start=q["berkeley_by_city"]["date_min"],
                          end=q["berkeley_by_city"]["date_max"], cities=q["berkeley_by_city"]["n_cities"]),
}
for p, v in q["drought_us_timeseries"]["splits"].items():
    summ[f"drought_us_{p}"] = dict(rows=v["rows"], start=v["date_min"], end=v["date_max"], counties=v["counties"])
pd.DataFrame(summ).T

,rows,cols,start,end,dup_ids,countries_raw,cities,counties
weather,170733,41,2024-05-16,2026-10-12,1,211,NaN,NaN
world_cities,50250.0,11.0,NaN,NaN,0.0,NaN,NaN,NaN
berkeley_by_city,8599212,7,1743-11-01,2013-09-01,NaN,NaN,3510,NaN
drought_us_train,19300680,NaN,2000-01-01,2016-12-31,NaN,NaN,NaN,3108
drought_us_validation,2268840,NaN,2017-01-01,2018-12-31,NaN,NaN,NaN,3108
drought_us_test,2271948,NaN,2019-01-01,2020-12-31,NaN,NaN,NaN,3108


In [7]:
print("Weather -9999 sentinels:", q["weather"]["sentinel_minus9999_counts"])
pd.Series(q["berkeley_by_city"]["missing_pct"]).rename("missing %").to_frame()

Weather -9999 sentinels: {'air_quality_Carbon_Monoxide': 1, 'air_quality_Sulphur_dioxide': 1}


,missing %
AverageTemperature,4.234
AverageTemperatureUncertainty,4.234


## 4. Scientific sources not used
NASA POWER, CSIC SPEIbase, MODIS MOD13A3/AppEEARS, OpenAQ and EM-DAT were named in the research
proposal, but the project was explicitly restricted to the four Kaggle datasets. Consequently no
observed SPEI, soil-moisture or NDVI is available for the capital cities, and **no synthetic
substitute was generated**. The dryness branch uses an observed atmospheric proxy instead (see
notebook 04), and the model is reported as a reduced-modality realisation.